# Figure 3: Intrinsic coupling of T cell memory phenotypes with V(D)J gene usage patterns

This notebook reproduces panels of **Figure 3** of the AIDA AIRR manuscript:

- **Fig. 3A** — Schematic (no code; metacells / V(D)J usage frequency feature space)
- **Fig. 3B, C** — V(D)J space UMAP of all T cells (B) and V/J gene enrichment heatmap by lineage (C)
- **Fig. 3D, E, F** — CD4+ V(D)J space: UMAP by cell type (D), Leiden cluster (E), per-cluster cell type composition (F)
- **Fig. 3G, H, I** — CD8+ V(D)J space: UMAP by cell type (G), Leiden cluster (H), per-cluster composition (I)
- **Fig. 3J, K** — V/J gene enrichment heatmaps for CD4+ (J) and CD8+ (K) V(D)J clusters
- **Fig. 3L** — V gene usage heatmap of CD8+ T cell subsets across ethnicities

The pipeline follows the Dandelion framework (Suo *et al.*, Nat Biotech 2024) augmented
with `milopy` to build per-metacell V(D)J usage profiles.

## Imports

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
from scipy.sparse import csr_matrix

import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns

import scanpy as sc
import anndata as ad
import dandelion as ddl
import milopy
import milopy.core as milo
import palantir
import sceleto2 as scjp

from sklearn.preprocessing import StandardScaler as standard, RobustScaler as robust
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text

%matplotlib inline
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, color_map='OrRd')

plt.rcParams['pdf.fonttype'] = 42
sns.set_style('ticks', {'axes.edgecolor': 'black', 'axes.edgewidth': 2,
                         "grid.color": "dimgray", "grid.linestyle": ":"})
sns.set_context("paper", font_scale=1.3, rc={'patch.linewidth': 1})


## Color palettes / category orders

In [ ]:
Ethnicity_order = ['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']
Ethnicity_colors = ["tomato", 'gold', "dodgerblue", "limegreen", "aquamarine", "orchid", 'gray']

T_anno2_order = [
    'T_CD4_Naive_SOX4', 'T_CD4_Naive', 'T_CD4_cTfh', 'T_CD4_Th1', 'T_CD4_Th2',
    'T_CD4_Th17', 'T_CD4_activated', 'T_CD4_CTL', 'T_CD4_Treg', 'T_CD4_IFN',
    'T_CD8_Naive_SOX4', 'T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB', 'T_CD8_KIR',
    'T_unc_MAIT', 'T_unc_gdT', 'T_unc_dnT'
]
T_anno2_colors = [
    "lightsteelblue", "#4a6fe3", "mediumorchid", "#bb7784", "darkorange",
    "#023fa5", "lightblue", "darkgreen", "#d6bcc0", "tan",
    "goldenrod", "yellowgreen", "lightcoral", "#d33f6a", "#11c638",
    "darkorchid", "#ef9708", "#0fcfc0",
]

cd4_color = [
    "lightsteelblue", "#4a6fe3", "mediumorchid", "#bb7784", "darkorange",
    "#023fa5", "lightblue", "darkgreen", "#d6bcc0", "tan",
]
cd8_colors = [
    "goldenrod", "yellowgreen", "lightcoral", "#d33f6a", "#11c638",
]


## Build dandelion contig object and merge with the T/NK AnnData

The contig file (`240716_TCR_ALL_AIRR.tsv`) is a single AIRR-format
table containing all TCR contigs across all donors and batches. We mask down to the donors
present in our AnnData, run `filter_contigs` to keep only paired αβ contigs and call clones.

In [ ]:
tdata = sc.read('data/06_250529_TDATA_anno2_meta.h5ad')

In [ ]:
sns.histplot(tdata.obs['PatientID'].value_counts(),bins=500)
#plt.xlim(0,500)
plt.xlabel('T cell number')
plt.ylabel('Patients number')
plt.axvline(x=np.percentile(tdata.obs['PatientID'].value_counts(), 5), color='r')
np.percentile(tdata.obs['PatientID'].value_counts(), 5)

pco = pd.DataFrame(tdata.obs['PatientID'].value_counts())
ptl = pco[tdata.obs['PatientID'].value_counts()>=633].index.tolist()
print('original {}'.format(len(tdata.obs['PatientID'].unique())))
print('after {}'.format(len(ptl)))


adata = tdata[tdata.obs['PatientID'].isin(ptl)]
adata = adata[~adata.obs['Ethnicity'].isna()]
 
adata = adata[adata.obs['Ethnicity']!='European']
adata = adata[~adata.obs['Ethnicity'].isna()]

In [ ]:
ptl = pco[pco['count'] >= np.percentile(pco['count'], 5)].index.tolist()
adata = tdata[tdata.obs['PatientID'].isin(ptl)]
adata = adata[adata.obs['Ethnicity'] != 'European']
adata = adata[~adata.obs['Ethnicity'].isna()]


In [ ]:
meta = adata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age','Sex','Ethnicity']]

In [ ]:
tcr = ddl.read_10x_airr('data/240716_TCR_ALL_AIRR.tsv')

In [ ]:
vdj, vdata = ddl.pp.filter_contigs(tcr, adata, library_type='tr-ab') 

In [ ]:
vdata = vdata[~vdata.obs.anno2.str.startswith('NK')]
vdata = vdata[~vdata.obs.anno2.str.startswith('T_gd')]

In [ ]:
vdata = vdata[~vdata.obs['v_call_VDJ'].isin(['No_contig'])]

In [ ]:
vdata.raw = vdata.copy()

# Whole T cell V(D)J space (Fig. 3B, C)

For each cell, the input feature is its V/J gene usage. We aggregate
transcriptomically similar cells into milo "metacells" and then compute V/J gene proportion vectors per metacell.

In [ ]:
ddata = ddl.tl.setup_vdj_pseudobulk(vdata,
                                    subsetby='anno2', 
                                    groups = vdata.obs['anno2'].cat.categories,
                                    mode = 'abT')

In [ ]:

 

sc.pp.neighbors(ddata, use_rep = "X_pca", n_neighbors = 50)

# use milo to sample neighbourhood
milo.make_nhoods(ddata)
# build neighbourhood ddata in ddata.uns['nhood_ddata']
milo.count_nhoods(ddata, sample_col='PatientID') # this step is needed to build ddata.uns['nhood_ddata'] and sample_col can be anything
# this step is needed for plotting below
milopy.utils.build_nhood_graph(ddata)
# assign neighbourhood celltype by majority voting
# results are in ddata.uns['nhood_ddata'].obs['nhood_annotation'] & ddata.uns['nhood_ddata'].obs['nhood_annotation_frac'] 
milopy.utils.annotate_nhoods(ddata, anno_col='anno2') 

sc.tl.umap(ddata)

min_overlap = 20
nhood_conn = ddata.uns['nhood_adata'].obsp['nhood_connectivities'].todense()
nhood_conn[nhood_conn < min_overlap] = 0  

ddata.uns['nhood_adata'].obsp['nhood_connectivities'] = sp.sparse.csr_matrix(nhood_conn)

# plot nhood on UMAP, but legend does not include line thickness and node size
sc.pl.embedding(ddata.uns['nhood_adata'], basis='X_milo_graph', sizes=list(ddata.uns['nhood_adata'].obs['Nhood_size']),
                color='nhood_annotation',
                neighbors_key='nhood')

In [ ]:
nhood_adata = ddl.tl.vdj_pseudobulk(ddata, pbs=ddata.obsm["nhoods"], obs_to_take=["anno2",'Ethnicity'],
                                  extract_cols=['v_call_VDJ_main', 'j_call_VDJ_main','v_call_VJ_main','j_call_VJ_main'])

sc.tl.pca(nhood_adata)
sc.pl.pca(nhood_adata, color=['anno2'])

sc.pp.neighbors(nhood_adata, random_state = 1712)
sc.tl.umap(nhood_adata, random_state = 1712)
sc.pl.umap(nhood_adata, color=['anno2'])


sc.pl.umap(nhood_adata, color=['anno2'])

sc.tl.leiden(nhood_adata, resolution=0.3)

sc.pl.umap(nhood_adata, color=['leiden'],  legend_loc='on data', legend_fontsize=10)

niso = pd.crosstab(nhood_adata.obs['leiden'], nhood_adata.obs['anno2'], normalize=0)

ax = niso.plot(kind='bar', stacked=True, 
               width=0.9,figsize=(3,5), ec='k')
plt.legend(loc=(1.01,0))
plt.xlabel('')
sns.despine()

sc.tl.rank_genes_groups(nhood_adata, groupby='leiden')

sc.pl.rank_genes_groups(nhood_adata)

In [ ]:
nhood_adata = nhood_adata[nhood_adata.obs['anno2']!='T_unc_gdT']

In [ ]:
nhood_adata.obs['anno2'] = nhood_adata.obs['anno2'].cat.reorder_categories([   'T_CD4_Naive_SOX4',  'T_CD4_Naive','T_CD4_cTfh','T_CD4_Th1', 'T_CD4_Th2',  'T_CD4_Th17','T_CD4_activated', 
               'T_CD4_CTL',       'T_CD4_Treg', 'T_CD4_IFN',
               'T_CD8_Naive_SOX4', 'T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB','T_CD8_KIR',
       'T_unc_MAIT','T_unc_dnT'])

In [ ]:
nhood_adata.uns['anno2_colors'] =   [
    "lightsteelblue",  # T_CD4_Naive_SOX4
    "#4a6fe3",  # T_CD4_Naive
    "mediumorchid",  # T_CD4_cTfh
    "#bb7784",  # T_CD4_Th1
    "darkorange",  # T_CD4_Th2
    "#023fa5",  # T_CD4_Th17
    "lightblue",  # T_CD4_activated
    "darkgreen",  # T_CD4_CTL
    "#d6bcc0",  # T_CD4_Treg
    "tan",  # T_CD4_IFN
    "goldenrod",  # T_CD8_Naive_SOX4
    "yellowgreen",  # T_CD8_Naive
    "lightcoral",  # T_CD8_TEM_GZMK
    "#d33f6a",  # T_CD8_TEM_GZMB
    "#11c638",  # T_CD8_KIR
    "#c6dec7",  # T_unc_MAIT
    
    "brown",  # T_unc_dnT
 
]

### Fig. 3B — UMAP of T cell metacells in V(D)J space

In [ ]:
scjp.us(nhood_adata, 'anno2', frameon=False)
plt.title('')

scjp.save_fig('Fig4_T_total_VDJ_space','UMAP',fig_folder='figures')

### Fig. 3C — Differential V/J genes per major lineage (CD4+, CD8+, MAIT)

In [ ]:
cd48 = nhood_adata[nhood_adata.obs['anno2'].str.startswith('T_CD')]
cd48.obs['anno0'] = 'T_CD8'
cd48.obs['anno0'] = np.where(cd48.obs['leiden'].str.startswith('0'), 'T_CD4', cd48.obs['anno0'])
sc.tl.rank_genes_groups(cd48, groupby='anno0')
sc.pl.rank_genes_groups(cd48)

In [ ]:
result =cd48.uns['rank_genes_groups']
groups = result['names'].dtype.names
wilcox_clusters = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj']})
wilcox_clusters.head(10)

In [ ]:
glist = wilcox_clusters[wilcox_clusters['T_CD8_l']<-0.5].sort_values(by='T_CD8_p', ascending=True).head(10)['T_CD8_n'].tolist() + wilcox_clusters[wilcox_clusters['T_CD8_l']>0.5].sort_values(by='T_CD8_p', ascending=True).head(10)['T_CD8_n'].tolist()

In [ ]:
cd48.layers['scale'] = cd48.X.copy()

In [ ]:
sc.pp.scale(cd48, max_value=2.9, layer='scale')

In [ ]:
sc.pl.heatmap(cd48,glist,
              'anno0',
         cmap='vlag', layer='scale', swap_axes=True, 
              vmax=2.9, vmin=-2.9, figsize=(7,5),
             )

In [ ]:
cd48 = nhood_adata.copy()
cd48.obs['anno0'] = cd48.obs['anno2'].copy()
cd48.obs['anno0'] = np.where(cd48.obs['anno2'].str.startswith('T_CD4'), 'T_CD4', cd48.obs['anno0'])
cd48.obs['anno0'] = np.where(cd48.obs['anno2'].str.startswith('T_CD8'), 'T_CD8', cd48.obs['anno0'])

sc.tl.rank_genes_groups(cd48, groupby='anno0')
sc.pl.rank_genes_groups(cd48)

In [ ]:
result =cd48.uns['rank_genes_groups']
groups = result['names'].dtype.names
wilcox_clusters = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj']})
wilcox_clusters.head(10)

In [ ]:
glist = wilcox_clusters[wilcox_clusters['T_CD4_l']>0.5].sort_values(by='T_CD4_p', ascending=True).head(5)['T_CD4_n'].tolist() + \
wilcox_clusters[wilcox_clusters['T_CD8_l']>0.5].sort_values(by='T_CD8_p', ascending=True).head(5)['T_CD8_n'].tolist() + \
wilcox_clusters[wilcox_clusters['T_unc_MAIT_l']>0.5].sort_values(by='T_unc_MAIT_p', ascending=True).head(5)['T_unc_MAIT_n'].tolist() + \
wilcox_clusters[wilcox_clusters['T_unc_dnT_l']>0.5].sort_values(by='T_unc_dnT_p', ascending=True).head(5)['T_unc_dnT_n'].tolist() 

In [ ]:
cd48.layers['scale'] = cd48.X.copy()

In [ ]:
sc.pp.scale(cd48, max_value=5, layer='scale')

In [ ]:
sc.pl.matrixplot(cd48,glist,
              'anno0',
         cmap='vlag', layer='scale', 
  vmax=1.6, vmin=-1.6, swap_axes=True,
    save='whole_T_vdj_matrixplot_by_anno0.pdf')

# CD4+ T cell V(D)J space (Fig. 3D, E, F, J)

In [ ]:
fdata = vdata[vdata.obs.anno2.str.startswith('T_CD4')]

fdata = fdata.raw.to_adata()

fdata.uns['log1p']['base'] = None

sc.pp.highly_variable_genes(fdata)



fdata.var.highly_variable = np.where(fdata.var.index.str.startswith('TRBV'), False, fdata.var.highly_variable)
fdata.var.highly_variable = np.where(fdata.var.index.str.startswith('TRAV'), False, fdata.var.highly_variable)
fdata.var.highly_variable = np.where(fdata.var.index.str.startswith('TRBJ'), False, fdata.var.highly_variable)
fdata.var.highly_variable = np.where(fdata.var.index.str.startswith('TRAJ'), False, fdata.var.highly_variable)
fdata.raw = fdata.copy()
sc.pp.scale(fdata, max_value=10)
sc.tl.pca(fdata)

sce.pp.harmony_integrate(fdata,'PatientID',adjusted_basis='X_pca')

sc.pp.neighbors(fdata, use_rep = "X_pca", n_neighbors = 50)
sc.tl.umap(fdata)

scjp.us(fdata,'anno2')

In [ ]:
dfdata = ddl.tl.setup_vdj_pseudobulk(fdata,
                                    subsetby='anno2', 
                                    groups = fdata.obs['anno2'].cat.categories,
                                    mode = 'abT')

In [ ]:
sc.pp.neighbors(dfdata, use_rep = "X_pca", n_neighbors = 50)

# use milo to sample neighbourhood
milo.make_nhoods(dfdata)
# build neighbourhood dfdata in dfdata.uns['nhood_dfdata']
milo.count_nhoods(dfdata, sample_col='PatientID') # this step is needed to build dfdata.uns['nhood_dfdata'] and sample_col can be anything
# this step is needed for plotting below
milopy.utils.build_nhood_graph(dfdata)
# assign neighbourhood celltype by majority voting
# results are in dfdata.uns['nhood_dfdata'].obs['nhood_annotation'] & dfdata.uns['nhood_dfdata'].obs['nhood_annotation_frac'] 
milopy.utils.annotate_nhoods(dfdata, anno_col='anno2') 

sc.tl.umap(dfdata)

min_overlap = 20
nhood_conn = dfdata.uns['nhood_adata'].obsp['nhood_connectivities'].todense()
nhood_conn[nhood_conn < min_overlap] = 0  

dfdata.uns['nhood_adata'].obsp['nhood_connectivities'] = sp.sparse.csr_matrix(nhood_conn)

# plot nhood on UMAP, but legend does not include line thickness and node size
sc.pl.embedding(dfdata.uns['nhood_adata'], basis='X_milo_graph', sizes=list(dfdata.uns['nhood_adata'].obs['Nhood_size']),
                color='nhood_annotation',
                neighbors_key='nhood')

In [ ]:
nhood_fdata = ddl.tl.vdj_pseudobulk(dfdata, pbs=dfdata.obsm["nhoods"], obs_to_take=["anno2",'Ethnicity'],
                                  extract_cols=['v_call_VDJ_main', 'j_call_VDJ_main','v_call_VJ_main','j_call_VJ_main'])

sc.tl.pca(nhood_fdata)
sc.pl.pca(nhood_fdata, color=['anno2'])

sc.pp.neighbors(nhood_fdata, random_state = 1712)
sc.tl.umap(nhood_fdata, random_state = 1712)
sc.pl.umap(nhood_fdata, color=['anno2'])

nhood_fdata = nhood_fdata[~nhood_fdata.obs.anno2.str.startswith('NK')]

nhood_fdata = nhood_fdata[~nhood_fdata.obs.anno2.str.startswith('T_gdT')]

sc.pl.umap(nhood_fdata, color=['anno2'])

sc.tl.leiden(nhood_fdata, resolution=0.3)

sc.pl.umap(nhood_fdata, color=['leiden'],  legend_loc='on data', legend_fontsize=10)

niso = pd.crosstab(nhood_fdata.obs['leiden'], nhood_fdata.obs['anno2'], normalize=0)

ax = niso.plot(kind='bar', stacked=True, 
               width=0.9,figsize=(3,5), ec='k')
plt.legend(loc=(1.01,0))
plt.xlabel('')
sns.despine()


In [ ]:
chood_fdata = nhood_fdata[nhood_fdata.obs.leiden!='3']

In [ ]:
chood_fdata.obs.anno2 = chood_fdata.obs.anno2.cat.reorder_categories(['T_CD4_Naive_SOX4',  'T_CD4_Naive','T_CD4_cTfh','T_CD4_Th1', 'T_CD4_Th2', 'T_CD4_activated',  'T_CD4_Th17','T_CD4_CTL',
       'T_CD4_Treg'])

In [ ]:
chood_fdata.uns['anno2_colors'] =  [
    "lightsteelblue",  # T_CD4_Naive_SOX4
    "#4a6fe3",  # T_CD4_Naive
    "mediumorchid",  # T_CD4_cTfh
    "#bb7784",  # T_CD4_Th1
    "darkorange",  # T_CD4_Th2
    "#023fa5",  # T_CD4_Th17
    "lightblue",  # T_CD4_activated
    "darkgreen",  # T_CD4_CTL
    "#d6bcc0",  # T_CD4_Treg
]

### Fig. 3D — UMAP of CD4+ metacells colored by cell type

In [ ]:
scjp.us(chood_fdata, 'anno2',  frameon=False)
plt.title('')
scjp.save_fig('Fig4_CD4_VDJ','UMAP',fig_folder='figures')

In [ ]:
sc.tl.leiden(chood_fdata, resolution=0.4)

In [ ]:
chood_fdata.obs['leiden'] = np.where(chood_fdata.obs['leiden']=='2', 'x', chood_fdata.obs['leiden'])
chood_fdata.obs['leiden'] = np.where(chood_fdata.obs['leiden']=='3', '2', chood_fdata.obs['leiden'])
chood_fdata.obs['leiden'] = np.where(chood_fdata.obs['leiden']=='x', '3', chood_fdata.obs['leiden'])

### Fig. 3E — UMAP of CD4+ metacells colored by Leiden cluster

In [ ]:
scjp.us(chood_fdata, 'leiden', frameon=False,
       legend_loc='on data', legend_fontsize=15)
plt.title('CD4+T cluster by VDJ usage', fontsize=15)
scjp.save_fig('Fig4_CD4_VDJ_leiden','UMAP',fig_folder='figures')

### Fig. 3F — Cell type composition per CD4+ V(D)J cluster

In [ ]:
niso = pd.crosstab( chood_fdata.obs['leiden'],chood_fdata.obs['anno2'], normalize=0)

ax = niso.plot(kind='bar', stacked=True, 
               width=0.9,figsize=(3,5), ec='k', color=chood_fdata.uns['anno2_colors'],
              alpha=0.8)
plt.legend(loc=(1.01,0))
plt.xlabel('')
sns.despine()
plt.savefig('figures/Fig4_CD4_leiden_anno2_prop.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    

### Fig. 3J — Top differential V/J genes per CD4+ V(D)J cluster

In [ ]:
sc.tl.rank_genes_groups(chood_fdata, groupby='leiden')
sc.pl.rank_genes_groups(chood_fdata)

In [ ]:
result =chood_fdata.uns['rank_genes_groups']
groups = result['names'].dtype.names
wilcox_clusters = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj']})
wilcox_clusters.head(10)

In [ ]:
glist = wilcox_clusters[wilcox_clusters['2_l']>0.5].sort_values(by='2_p', ascending=True).head(10)['2_n'].tolist() + \
wilcox_clusters[wilcox_clusters['3_l']>0.5].sort_values(by='3_p', ascending=True).head(10)['3_n'].tolist() 

In [ ]:
chood_fdata.layers['scale'] = chood_fdata.X.copy()

In [ ]:
sc.pp.scale(chood_fdata,   max_value=5, layer='scale')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(chood_fdata, layer='scale', cmap='vlag', vmax=1.9, vmin=-1.9,
                                  n_genes=5, swap_axes=True, figsize=(4,9), dendrogram=False,
                                                 save='CD4_vdj_matrixplot_by_leiden.pdf')

# CD8+ T cell V(D)J space (Fig. 3G, H, I, K)

In [ ]:
edata = vdata[vdata.obs.anno2.str.startswith('T_CD8')]

In [ ]:
dedata = ddl.tl.setup_vdj_pseudobulk(edata,
                                    subsetby='anno2', 
                                    groups = edata.obs['anno2'].cat.categories,
                                    mode = 'abT')

sc.pp.neighbors(dedata, use_rep = "X_pca", n_neighbors = 50)

# use milo to sample neighbourhood
milo.make_nhoods(dedata)
# build neighbourhood dedata in dedata.uns['nhood_dedata']
milo.count_nhoods(dedata, sample_col='PatientID') # this step is needed to build dedata.uns['nhood_dedata'] and sample_col can be anything
# this step is needed for plotting below
milopy.utils.build_nhood_graph(dedata)
# assign neighbourhood celltype by majority voting
# results are in dedata.uns['nhood_dedata'].obs['nhood_annotation'] & dedata.uns['nhood_dedata'].obs['nhood_annotation_frac'] 
milopy.utils.annotate_nhoods(dedata, anno_col='anno2') 

sc.tl.umap(dedata)

min_overlap = 20
nhood_conn = dedata.uns['nhood_adata'].obsp['nhood_connectivities'].todense()
nhood_conn[nhood_conn < min_overlap] = 0  

dedata.uns['nhood_adata'].obsp['nhood_connectivities'] = sp.sparse.csr_matrix(nhood_conn)

# plot nhood on UMAP, but legend does not include line thickness and node size
sc.pl.embedding(dedata.uns['nhood_adata'], basis='X_milo_graph', sizes=list(dedata.uns['nhood_adata'].obs['Nhood_size']),
                color='nhood_annotation',
                neighbors_key='nhood')

In [ ]:
nhood_edata = ddl.tl.vdj_pseudobulk(dedata, pbs=dedata.obsm["nhoods"], obs_to_take=["anno2",'Ethnicity'],
                                  extract_cols=['v_call_VDJ_main', 'j_call_VDJ_main','v_call_VJ_main','j_call_VJ_main'])

sc.tl.pca(nhood_edata)
sc.pl.pca(nhood_edata, color=['anno2'])

sc.pp.neighbors(nhood_edata, random_state = 1712)
sc.tl.umap(nhood_edata, random_state = 1712)
sc.pl.umap(nhood_edata, color=['anno2'])

nhood_edata = nhood_edata[~nhood_edata.obs.anno2.str.startswith('NK')]
nhood_edata = nhood_edata[~nhood_edata.obs.anno2.str.startswith('T_unc_gdT')]

sc.pl.umap(nhood_edata, color=['anno2'])

sc.tl.leiden(nhood_edata, resolution=0.3)

sc.pl.umap(nhood_edata, color=['leiden'],  legend_loc='on data', legend_fontsize=10)

niso = pd.crosstab(nhood_edata.obs['leiden'], nhood_edata.obs['anno2'], normalize=0)

ax = niso.plot(kind='bar', stacked=True, 
               width=0.9,figsize=(3,5), ec='k')
plt.legend(loc=(1.01,0))
plt.xlabel('')
sns.despine()


In [ ]:
chood_edata = nhood_edata.copy()

In [ ]:
nhood_edata.obs['anno2'] = nhood_edata.obs['anno2'].cat.reorder_categories(['T_CD8_Naive_SOX4','T_CD8_Naive','T_CD8_TEM_GZMK',
                                                                            'T_CD8_TEM_GZMB','T_CD8_KIR'])

In [ ]:
chood_edata.obs['anno2'] = chood_edata.obs['anno2'].cat.reorder_categories(['T_CD8_Naive_SOX4','T_CD8_Naive','T_CD8_TEM_GZMK',
                                                                            'T_CD8_TEM_GZMB','T_CD8_KIR'])

In [ ]:
chood_edata.uns['anno2_colors'] =      [  "goldenrod",  # T_CD8_Naive_SOX4
    "yellowgreen",  # T_CD8_Naive
    "lightcoral",  # T_CD8_TEM_GZMK
    "#d33f6a",  # T_CD8_TEM_GZMB
    "#11c638",  # T_CD8_KIR
]

### Fig. 3G — UMAP of CD8+ metacells colored by cell type

In [ ]:
scjp.us(chood_edata, 'anno2', s=30, frameon=False)
plt.title('VDJ usage neighborhood space')
scjp.save_fig('Fig4_CD8_VDJ','UMAP',fig_folder='figures')

In [ ]:
sc.tl.leiden(chood_edata, resolution=0.5)

### Fig. 3H — UMAP of CD8+ metacells colored by Leiden cluster

In [ ]:
scjp.us(chood_edata, 'leiden', s=30, frameon=False)
plt.title('VDJ usage neighborhood space')
scjp.save_fig('Fig4_CD8_VDJ_leiden','UMAP',fig_folder='figures')

### Fig. 3I — Cell type composition per CD8+ V(D)J cluster

In [ ]:
niso = pd.crosstab(chood_edata.obs['leiden'], chood_edata.obs['anno2'], normalize=0)

In [ ]:
ax = niso.plot(kind='bar', stacked=True, 
               width=0.9,figsize=(3,5), ec='k', color=chood_edata.uns['anno2_colors'],
              alpha=0.8)
plt.legend(loc=(1.01,0))
plt.xlabel('')
sns.despine()
plt.savefig('figures/Fig4_CD8_leiden_anno2_prop.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    

### Fig. 3K — Top differential V/J genes per CD8+ V(D)J cluster

In [ ]:
sc.tl.rank_genes_groups(chood_edata, groupby='leiden')
sc.pl.rank_genes_groups(chood_edata)

In [ ]:
result =chood_edata.uns['rank_genes_groups']
groups = result['names'].dtype.names
wilcox_clusters = pd.DataFrame(
{group + '_' + key[:1]: result[key][group]
    for group in groups for key in ['names', 'logfoldchanges', 'pvals_adj']})
wilcox_clusters.head(10)

In [ ]:
glist = wilcox_clusters[wilcox_clusters['2_l']>0.5].sort_values(by='2_p', ascending=True).head(10)['2_n'].tolist() + \
wilcox_clusters[wilcox_clusters['3_l']>0.5].sort_values(by='3_p', ascending=True).head(10)['3_n'].tolist() 

In [ ]:
chood_edata.layers['scale'] = chood_edata.X.copy()

In [ ]:
sc.pp.scale(chood_edata,   max_value=5, layer='scale')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(chood_edata, layer='scale', cmap='vlag', vmax=1.9, vmin=-1.9,
                                  n_genes=5, swap_axes=True, figsize=(4,9), dendrogram=False,
                                                 save='CD8_vdj_matrixplot_by_leiden.pdf')

# Fig. 3L — V gene usage heatmap of CD8+ subsets across ethnicities

For each (cell type, ethnicity) combination, we compute mean per-donor V gene proportions and
standard-scale across cell types so that ethnic-level differences within a subset become visible.
Genes with overall mean proportion < 0.5% are dropped to reduce noise.

In [ ]:
dedata.obs['Ethnicity'] = dedata.obs['Ethnicity'].cat.reorder_categories(Ethnicity_order)

In [ ]:
tf = dedata.obs[~(dedata.obs['anno2'].str.startswith('T_CD8_Naive_S'))] 
tf.anno2 = tf.anno2.astype('str')
tf.anno2 = tf.anno2.astype('category')
tf.anno2 = tf.anno2.cat.reorder_categories(['T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB','T_CD8_KIR'])

tf = tf[tf.PatientID.isin(tf['PatientID'].value_counts()[tf['PatientID'].value_counts()>200].index)]

ttf = pd.crosstab([tf['anno2'],tf['PatientID']], tf['v_call_abT_VDJ_main'], normalize=0)
mlist = ttf.mean()[ttf.mean()>0.005].index
mf = ttf.copy()

 


mf = mf.reset_index().merge(meta.reset_index()[['PatientID','Ethnicity']], left_on='PatientID', right_on='PatientID',how='left' )

mmf = mf.groupby(['anno2','Ethnicity']).mean()

mmf = pd.DataFrame(standard().fit_transform(mmf.T), index = mmf.T.index, columns=mmf.T.columns)
mmf = mmf.T 

mmf = mmf[mlist]

plt.figure(figsize=(len(mmf.columns)/3.1,6))
fig = sns.heatmap(mmf, cmap='RdYlBu_r',  linecolor='k', linewidth=0.3,
                  vmax=2.7, vmin=-2.7, )

cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
cbar.set_ylabel('Standard scale', size=15)
plt.xlabel('')
plt.ylabel('')

fig.axhline(y = 0, color='k',linewidth = 3) 
fig.axhline(y = mmf.shape[0], color = 'k', 
            linewidth = 3) 

fig.axvline(x = 0, color = 'k', 
            linewidth = 3) 

fig.axvline(x = mmf.shape[1],  
             color = 'k', linewidth = 3) 

fig.axhline(y = 6, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 12, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 18, color = 'k', 
            linewidth = 1.5, linestyle='--')

plt.savefig('figures/Fig3_T_CD8_TRBV_heatmap.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    

### Per-gene point plot (top up- and down-regulated genes in CD8+ KIR+ T cells)

In [ ]:
tf = dedata.obs[~(dedata.obs['anno2'].str.startswith('T_CD8_Naive_S'))] 
tf.anno2 = tf.anno2.astype('str')
tf.anno2 = tf.anno2.astype('category')
tf.anno2 = tf.anno2.cat.reorder_categories(['T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB','T_CD8_KIR'])

tf = tf[tf.PatientID.isin(tf['PatientID'].value_counts()[tf['PatientID'].value_counts()>200].index)]

ttf = pd.crosstab([tf['anno2'],tf['PatientID']], tf['v_call_abT_VDJ_main'], normalize=0)
mlist = ttf.mean()[ttf.mean()>0.005].index
mf = ttf.copy()
mf = mf.reset_index().merge(meta.reset_index()[['PatientID','Ethnicity']], left_on='PatientID', right_on='PatientID',how='left' )

mmf = mf.groupby(['anno2','Ethnicity']).mean()

mmf = pd.DataFrame(standard().fit_transform(mmf.T), index = mmf.T.index, columns=mmf.T.columns)
mmf = mmf.T 

mmf = mmf[mlist]

In [ ]:
# --- 0) select Top10 up / Top10 down by signed mean diff (KIR vs others) ---
# mean_by_cell: per-donor mean at anno2 level
mean_by_cell = ttf[mlist].groupby(level=0).mean()

# Non-KIR mean
other_mean = mean_by_cell.drop(index='T_CD8_KIR').mean(axis=0)

# KIR - Others (sign-preserving)
diff_signed = mean_by_cell.loc['T_CD8_KIR'] - other_mean

# Top10 up (largest positive), Top10 down (smallest negative)
pos10 = diff_signed.sort_values(ascending=False).head(10).index.tolist()
neg10 = diff_signed.sort_values(ascending=True).head(10).index.tolist()

# facet order: 10 up + 10 down (20 total -> 4x5)
var_order = pos10 + neg10

# --- 1) prepare long-format data (reuse mf) ---
df_long = mf.melt(
    id_vars=['anno2', 'Ethnicity'],
    value_vars=var_order,
    var_name='gene',
    value_name='proportion'
)

# --- 2) 4×5 facet pointplot ---
g = sns.catplot(
    data=df_long,
    x='anno2', y='proportion',
    hue='Ethnicity',
    col='gene', col_order=var_order,   # fix facet order
    col_wrap=5,                        # 5 columns -> 4 rows
    kind='point',
    hue_order=Ethnicity_order,
    palette=Ethnicity_colors,
    dodge=0.5,
    capsize=0.1,
    errwidth=1,
    sharey=False,
    errorbar='se',
    height=3.5,
    aspect=0.6,
)

# tidy axes / spines
for ax in g.axes.flatten():
    sns.despine(ax=ax)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90)

# add up/down indicator to title
g.set_titles("{col_name}")  # show only the gene name in title
pos_set = set(pos10)
for ax in g.axes.flatten():
    ttl = ax.get_title()
    arrow = "↑" if ttl in pos_set else "↓"
    ax.set_title(f"{ttl} (KIR{arrow})", fontsize=15)

# legend / layout
g.fig.tight_layout()
g.add_legend(title='Ethnicity', bbox_to_anchor=(1, 0.5), loc='center left')

plt.savefig('figures/Fig3_T_CD8_TRBV_KIRpos10_neg10_pointfacet.pdf',
            dpi=300, format='pdf', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# 1. Compute mean TRBV proportions per cell type
mean_by_cell = ttf[mlist].groupby(level=0).mean()

# 2. Compute the average TRBV usage across all non‑KIR T cell types
other_cell_types = mean_by_cell.index.drop('T_CD8_KIR')
other_mean = mean_by_cell.loc[other_cell_types].mean(axis=0)

# 3. Compute absolute differences between KIR T cells and the others
diff = (mean_by_cell.loc['T_CD8_KIR'] - other_mean).abs()

# 4. Select the top 9 genes with the largest difference
top9_genes = diff.sort_values(ascending=False).head(25).index.tolist()

# 5. Use these 9 genes for your catplot
var_order = top9_genes

# Now you can proceed with melting and plotting using var_order[:9]
df_long = mf.melt(
    id_vars=['anno2', 'Ethnicity'],
    value_vars=var_order,
    var_name='gene',
    value_name='proportion'
)

g = sns.catplot(
    data=df_long,
    x='anno2', y='proportion',
    hue='Ethnicity',
    col='gene',
    col_wrap=5,
    kind='point',
    hue_order=Ethnicity_order,
    palette=Ethnicity_colors,
    dodge=0.5,
    capsize=0.1,
    errwidth=1,
    sharey=False,
    errorbar='se',
    height=3.5,
    aspect=0.6,
)

for ax in g.axes.flatten():
    sns.despine(ax=ax)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90)

g.fig.tight_layout()
g.add_legend(title='Ethnicity', bbox_to_anchor=(1, 0.5), loc='center left')

plt.savefig('figures/Fig3_T_CD8_TRBV_diff_top_facet.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    
plt.show()

In [ ]:
tf = dedata.obs[~(dedata.obs['anno2'].str.startswith('T_CD8_Naive_S'))] 
tf.anno2 = tf.anno2.astype('str')
tf.anno2 = tf.anno2.astype('category')
tf.anno2 = tf.anno2.cat.reorder_categories(['T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB','T_CD8_KIR'])

tf = tf[tf.PatientID.isin(tf['PatientID'].value_counts()[tf['PatientID'].value_counts()>200].index)]

ttf = pd.crosstab([tf['anno2'],tf['PatientID']], tf['v_call_abT_VJ_main'], normalize=0)
mlist = ttf.mean()[ttf.mean()>0.005].index
mf = ttf.copy()

mf = mf.reset_index().merge(meta.reset_index()[['PatientID','Ethnicity']], left_on='PatientID', right_on='PatientID',how='left' )

mmf = mf.groupby(['anno2','Ethnicity']).mean()

mmf = pd.DataFrame(standard().fit_transform(mmf.T), index = mmf.T.index, columns=mmf.T.columns)
mmf = mmf.T 

mmf = mmf[mlist]

plt.figure(figsize=(len(mmf.columns)/3.1,6))
fig = sns.heatmap(mmf, cmap='RdYlBu_r',  linecolor='k', linewidth=0.3,
                  vmax=2.6, vmin=-2.6, )

cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
cbar.set_ylabel('Standard scale', size=15)
plt.xlabel('')
plt.ylabel('')

fig.axhline(y = 0, color='k',linewidth = 3) 
fig.axhline(y = mmf.shape[0], color = 'k', 
            linewidth = 3) 

fig.axvline(x = 0, color = 'k', 
            linewidth = 3) 

fig.axvline(x = mmf.shape[1],  
             color = 'k', linewidth = 3) 

fig.axhline(y = 6, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 12, color = 'k', 
            linewidth = 1.5, linestyle='--') 
fig.axhline(y = 18, color = 'k', 
            linewidth = 1.5, linestyle='--')

plt.savefig('figures/Fig3_T_CD8_TRAV_heatmap.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')    

In [ ]:
# 1. Compute mean TRBV proportions per cell type
mean_by_cell = ttf[mlist].groupby(level=0).mean()

# 2. Compute the average TRBV usage across all non‑KIR T cell types
other_cell_types = mean_by_cell.index.drop('T_CD8_KIR')
other_mean = mean_by_cell.loc[other_cell_types].mean(axis=0)

# 3. Compute absolute differences between KIR T cells and the others
diff = (mean_by_cell.loc['T_CD8_KIR'] - other_mean).abs()

# 4. Select the top 9 genes with the largest difference
top9_genes = diff.sort_values(ascending=False).head(9).index.tolist()

# 5. Use these 9 genes for your catplot
var_order = top9_genes

# Now you can proceed with melting and plotting using var_order[:9]
df_long = mf.melt(
    id_vars=['anno2', 'Ethnicity'],
    value_vars=var_order,
    var_name='gene',
    value_name='proportion'
)

g = sns.catplot(
    data=df_long,
    x='anno2', y='proportion',
    hue='Ethnicity',
    col='gene',
    col_wrap=3,
    kind='point',
    hue_order=Ethnicity_order,
    palette=Ethnicity_colors,
    dodge=0.5,
    capsize=0.15,
    errwidth=1,
    sharey=False,
    errorbar='se',
    height=3.5,
    aspect=0.7, legend=False
)
for ax in g.axes.flatten():
    # Remove top and right spines if desired
    sns.despine(ax=ax)
    # Add vertical dashed grid lines
    ax.grid(axis='x', linestyle='--', which='major', color='grey', alpha=0.7)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=90)

g.fig.tight_layout()
g.add_legend(title='Ethnicity', bbox_to_anchor=(0.9, 0.2), loc='center left')
plt.show()